# <font color='blue'> Baseline Linear Regression </font>

## <font color='orange'> Loading Data </font>

In [2]:
# loading data
import pandas as pd

# Load the Excel file into a DataFrame
df = pd.read_csv('../data/used_cars.xls')

# Display the first few rows
df.head()

,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


## <font color='orange'> Using K-Fold Data Splitting </font>

In [11]:
import pandas as pd
import category_encoders as ce
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error, r2_score
from sklearn.pipeline import Pipeline

# separate target (y) and features (X)
y = df['selling_price']

# drop columns to be completely ignored
X = df.drop(columns=['selling_price'])

# defining categorical columns
categorical_cols = ['name', 'fuel', 'transmission', 'seller_type', 'owner']

# using pipeline to combine Target Encoding and the model
# this will prevent data leakage
model_pipeline = Pipeline(steps=[
    ('encoder', ce.TargetEncoder(cols=categorical_cols)),
    ('regressor', LinearRegression())
])

# Initialize K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# evaluate using cross_val_score
scores = cross_val_score(model_pipeline, X, y, cv=kf, scoring='neg_root_mean_squared_error')
rmse_scores = -scores

r2_scores = cross_val_score(model_pipeline, X, y, cv=kf, scoring="r2")

# performance summary
print(f"Average RMSE: {np.mean(rmse_scores):.2f} (+/- {np.std(rmse_scores):.2f})")
print(f"Average R² score: {np.mean(r2_scores):.3f} (+/- {np.std(r2_scores):.3f})")

Average RMSE: 355176.02 (+/- 38292.17)
Average R² score: 0.617 (+/- 0.089)


## <font color='orange'> Using Test-Train Split </font>

In [15]:
# Step 1: Separate target (y) and features (X)
y = df['selling_price']
X = df.drop(columns=['selling_price'])

# Step 2: One-hot encode categorical variables
X_encoded = pd.get_dummies(X, columns=['fuel', 'transmission', 'seller_type', 'name', 'owner'], drop_first=True)

# Step 3: Split into training and testing sets (e.g., 80% train, 20% test)
# random_state ensures you get the exact same split every time you run it
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

# Step 4: Initialize and train your model
model = LinearRegression()
model.fit(X_train, y_train)

# Step 5: Make predictions on the unseen test data
predictions = model.predict(X_test)

# Step 6: Evaluate the performance
rmse = root_mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Test RMSE: {rmse:.2f}")
print(f"Test R² Score: {r2:.2f}")

Test RMSE: 395243.69
Test R² Score: 0.49


### <font color='blue'> The K-Fold method clearly performs better than a simple Test-Train Split. </font>

## <font color='orange'> Ridge Regression (Using Test-Train Split) </font>

In [20]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score

# set target and feature columns
y = df['selling_price']
X = df.drop(columns=['selling_price'])

X_encoded = pd.get_dummies(X, columns=['fuel', 'transmission', 'seller_type', 'name', 'owner'], drop_first=True)

# scale the features so that the ridge penaly affects them equally
# this is really important for Ridge, Lasso and ElasticNet methods
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)

# split into train-test datasets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# initialise the ridge method
ridge = Ridge(alpha=1.0)
ridge.fit(X_train, y_train)
pred_basic = ridge.predict(X_test)


rmse = root_mean_squared_error(y_test, pred_basic)
r2 = r2_score(y_test, pred_basic)

print(f"Test RMSE: {rmse:.2f}")
print(f"Test R² Score: {r2:.2f}")

print("MSE (alpha = 1.0):", mean_squared_error(y_test, pred_basic))
print("Coefficients (alpha = 1.0):", ridge.coef_)

param_grid = {"alpha": [0.001, 0.01, 0.1, 1, 10, 100, 500]}
grid = GridSearchCV(Ridge(), param_grid, cv=5, scoring="neg_mean_squared_error")

grid.fit(X_train, y_train)
best_ridge = grid.best_estimator_
pred_best = best_ridge.predict(X_test)

print("Best alpha selected:", grid.best_params_["alpha"])
print("MSE (best alpha):", mean_squared_error(y_test, pred_best))
print("Coefficients (best alpha):", best_ridge.coef_)

Test RMSE: 358161.94
Test R² Score: 0.58
MSE (alpha = 1.0): 128279975630.41557
Coefficients (alpha = 1.0): [110865.61204798 -25698.39868037  18703.40875793 ...  -9410.53633042
   5421.41976378  -7394.99071017]
Best alpha selected: 100
MSE (best alpha): 123571711425.61972
Coefficients (best alpha): [103195.10574977 -26843.51166886  50020.80809342 ... -11468.83866987
   5731.71937335  -8465.82590948]


## <font color='orange'> Ridge Regression (Using K-Fold) </font>

In [21]:
import numpy as np
import pandas as pd
import category_encoders as ce
from sklearn.pipeline import Pipeline
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, root_mean_squared_error, r2_score

# --- Assuming df is already loaded ---
# y = df['selling_price']
# X = df.drop(columns=['selling_price'])

# Train-test split for the standalone model and grid search sections
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# defining categorical columns
categorical_cols = ['name', 'fuel', 'transmission', 'seller_type', 'owner']

# 1. Pipeline for Linear Regression with Target Encoding and Standardization
model_pipeline = Pipeline(steps=[
    ('encoder', ce.TargetEncoder(cols=categorical_cols)),
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

# Initialize K-Fold
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# evaluate using cross_val_score
scores = cross_val_score(model_pipeline, X, y, cv=kf, scoring='neg_root_mean_squared_error')
rmse_scores = -scores

r2_scores = cross_val_score(model_pipeline, X, y, cv=kf, scoring="r2")

# performance summary
print(f"Average RMSE: {np.mean(rmse_scores):.2f} (+/- {np.std(rmse_scores):.2f})")
print(f"Average R² score: {np.mean(r2_scores):.3f} (+/- {np.std(r2_scores):.3f})")

# 2. Pipeline for Ridge Regression (with Standardization and Target Encoding)
ridge_pipeline = Pipeline(steps=[
    ('encoder', ce.TargetEncoder(cols=categorical_cols)),
    ('scaler', StandardScaler()),
    ('regressor', Ridge(alpha=1.0))
])

ridge_pipeline.fit(X_train, y_train)
pred_basic = ridge_pipeline.predict(X_test)

rmse = root_mean_squared_error(y_test, pred_basic)
r2 = r2_score(y_test, pred_basic)

print(f"\nTest RMSE (alpha = 1.0): {rmse:.2f}")
print(f"Test R² Score (alpha = 1.0): {r2:.2f}")
print("MSE (alpha = 1.0):", mean_squared_error(y_test, pred_basic))
print("Coefficients (alpha = 1.0):", ridge_pipeline.named_steps['regressor'].coef_)

# 3. GridSearchCV Pipeline for Ridge Optimization
param_grid = {"regressor__alpha": [0.001, 0.01, 0.1, 1, 10, 100, 500]}

grid_pipeline = Pipeline(steps=[
    ('encoder', ce.TargetEncoder(cols=categorical_cols)),
    ('scaler', StandardScaler()),
    ('regressor', Ridge())
])

grid = GridSearchCV(grid_pipeline, param_grid, cv=5, scoring="neg_mean_squared_error")
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
pred_best = best_model.predict(X_test)

# Extract best alpha parameter name format from pipeline
best_alpha = grid.best_params_["regressor__alpha"]

print(f"\nBest alpha selected: {best_alpha}")
print("MSE (best alpha):", mean_squared_error(y_test, pred_best))
print("Coefficients (best alpha):", best_model.named_steps['regressor'].coef_)

Average RMSE: 355176.02 (+/- 38292.17)
Average R² score: 0.617 (+/- 0.089)

Test RMSE (alpha = 1.0): 406070.29
Test R² Score (alpha = 1.0): 0.46
MSE (alpha = 1.0): 164893076595.25668
Coefficients (alpha = 1.0): [455622.4190388   54044.76053005 -24114.80667179  14739.18768755
  -6743.67271505  79505.95768417  10582.57111623]

Best alpha selected: 500
MSE (best alpha): 157340779766.45255
Coefficients (best alpha): [378905.75097254  62654.28941147 -26444.46218292  33587.94103441
   4653.3248951   98148.81154676  13773.45526191]


## <font color='orange'> Lasso Regression (Using K-Fold) </font>

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load Dataset
df = pd.read_csv('../data/used_cars.xls')

# 2. Separate Features and Target
X = df.drop('selling_price', axis=1)
y = df['selling_price']

# 3. Encode Categorical Features globally
X = pd.get_dummies(X, drop_first=True)

# 4. Setup K-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = []
rmse_scores = []
best_alphas = []

# 5. Iterating through K-Folds
for fold, (train_index, test_index) in enumerate(kf.split(X)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Feature Scaling fit strictly on training fold data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # LassoCV training (Fixed line)
    lasso_cv = LassoCV(cv=5, random_state=42, max_iter=10000)
    lasso_cv.fit(X_train_scaled, y_train)
    
    # Predict and evaluate
    y_pred = lasso_cv.predict(X_test_scaled)
    
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    r2_scores.append(r2)
    rmse_scores.append(rmse)
    best_alphas.append(lasso_cv.alpha_)
    
    print(f"Fold {fold + 1} - Alpha: {lasso_cv.alpha_:.4f} | R2: {r2:.4f} | RMSE: {rmse:.4f}")

# 6. Summary Statistics
print("\n--- K-Fold Cross-Validation Summary ---")
print(f"Average R2 Score: {np.mean(r2_scores):.4f} (+/- {np.std(r2_scores):.4f})")
print(f"Average RMSE: {np.mean(rmse_scores):.4f} (+/- {np.std(rmse_scores):.4f})")
print(f"Average Optimal Alpha: {np.mean(best_alphas):.4f}")

Fold 1 - Alpha: 417.3064 | R2: 0.5710 | RMSE: 361810.0793
Fold 2 - Alpha: 433.3604 | R2: 0.8212 | RMSE: 236585.5723
Fold 3 - Alpha: 460.2542 | R2: 0.8490 | RMSE: 226231.0215
Fold 4 - Alpha: 403.2836 | R2: 0.7823 | RMSE: 273269.8603
Fold 5 - Alpha: 431.2185 | R2: 0.6970 | RMSE: 335975.7361

--- K-Fold Cross-Validation Summary ---
Average R2 Score: 0.7441 (+/- 0.1006)
Average RMSE: 286774.4539 (+/- 53699.0062)
Average Optimal Alpha: 429.0846


## <font color='orange'> Lasso Regression (Test-Train Split) </font>

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load Dataset
df = pd.read_csv('../data/used_cars.xls')

# 2. Separate Features and Target
X = df.drop('selling_price', axis=1)
y = df['selling_price']

# 3. Encode Categorical Features to Numeric Data
X = pd.get_dummies(X, drop_first=True)

# 4. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Initialize and Train LassoCV (Fixed line)
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=10000)
lasso_cv.fit(X_train_scaled, y_train)

# 7. Model Evaluation
y_pred = lasso_cv.predict(X_test_scaled)

print("--- Train-Test Split Results ---")
print(f"Optimal Alpha Selected: {lasso_cv.alpha_:.4f}")
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

--- Train-Test Split Results ---
Optimal Alpha Selected: 479.8010
R2 Score: 0.5711
RMSE: 361789.2855


## <font color='orange'> ElasticNet Regression (Test-Train Split) </font>

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load Dataset
df = pd.read_csv('../data/used_cars.xls')

# 2. Separate Features and Target
X = df.drop('selling_price', axis=1)
y = df['selling_price']

# 3. Encode Categorical Features to Numeric Data
X = pd.get_dummies(X, drop_first=True)

# 4. Train-Test Split (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5. Feature Scaling (Essential for ElasticNet regularization)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 6. Initialize and Train ElasticNetCV
# l1_ratio controls the balance between L1 (Lasso) and L2 (Ridge) penalties
# l1_ratio = 1 is pure Lasso, l1_ratio = 0 is pure Ridge
elastic_cv = ElasticNetCV(
    l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0], 
    cv=5, 
    random_state=42, 
    max_iter=10000
)
elastic_cv.fit(X_train_scaled, y_train)

# 7. Model Evaluation
y_pred = elastic_cv.predict(X_test_scaled)

print("--- ElasticNet Train-Test Split Results ---")
print(f"Optimal Alpha Selected: {elastic_cv.alpha_:.4f}")
print(f"Optimal L1-Ratio Selected: {elastic_cv.l1_ratio_:.2f}")
print(f"R2 Score: {r2_score(y_test, y_pred):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}")

# Optional: View non-zero coefficients
coefficients = pd.DataFrame({'Feature': X.columns, 'Coefficient': elastic_cv.coef_})
print("\nTop Non-Zero Coefficients:")
print(coefficients[coefficients['Coefficient'] != 0].head(10))

--- ElasticNet Train-Test Split Results ---
Optimal Alpha Selected: 479.8010
Optimal L1-Ratio Selected: 1.00
R2 Score: 0.5711
RMSE: 361789.2855

Top Non-Zero Coefficients:
                                     Feature    Coefficient
0                                       year  118827.293554
1                                  km_driven  -25956.834626
2           name_Ambassador Classic 2000 Dsz    -526.339890
3  name_Ambassador Grand 1800 ISZ MPFI PW CL    2377.582069
4                      name_Audi A4 1.8 TFSI    9974.810929
5                       name_Audi A4 2.0 TDI    8483.224047
6  name_Audi A4 2.0 TDI 177 Bhp Premium Plus    5016.809351
7               name_Audi A4 3.0 TDI Quattro   42916.528347
8            name_Audi A4 30 TFSI Technology   54725.767495
9                name_Audi A4 35 TDI Premium   24192.373735


## <font color='orange'> ElasticNet Regression (K-Fold) </font>

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.linear_model import ElasticNetCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# 1. Load Dataset
df = pd.read_csv('../data/used_cars.xls')

# 2. Separate Features and Target
X = df.drop('selling_price', axis=1)
y = df['selling_price']

# 3. Encode Categorical Features globally
X = pd.get_dummies(X, drop_first=True)

# 4. Setup K-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

r2_scores = []
rmse_scores = []
best_alphas = []
best_l1_ratios = []

# 5. Iterating through K-Folds
for fold, (train_index, test_index) in enumerate(kf.split(X)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]
    
    # Feature Scaling fit strictly on training fold data
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    # ElasticNetCV training
    # l1_ratio= [.1, .5, .7, .9, .95, .99, 1] tests a mix of L1 (Lasso) and L2 (Ridge)
    elastic_cv = ElasticNetCV(
        l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95, 0.99, 1.0], 
        cv=5, 
        random_state=42, 
        max_iter=10000
    )
    elastic_cv.fit(X_train_scaled, y_train)
    
    # Predict and evaluate
    y_pred = elastic_cv.predict(X_test_scaled)
    
    r2 = r2_score(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    r2_scores.append(r2)
    rmse_scores.append(rmse)
    best_alphas.append(elastic_cv.alpha_)
    best_l1_ratios.append(elastic_cv.l1_ratio_)
    
    print(f"Fold {fold + 1} - Alpha: {elastic_cv.alpha_:.4f} | L1-Ratio: {elastic_cv.l1_ratio_:.2f} | R2: {r2:.4f} | RMSE: {rmse:.4f}")

# 6. Summary Statistics
print("\n--- ElasticNetCV K-Fold Cross-Validation Summary ---")
print(f"Average R2 Score: {np.mean(r2_scores):.4f} (+/- {np.std(r2_scores):.4f})")
print(f"Average RMSE: {np.mean(rmse_scores):.4f} (+/- {np.std(rmse_scores):.4f})")
print(f"Average Optimal Alpha: {np.mean(best_alphas):.4f}")
print(f"Average Optimal L1-Ratio: {np.mean(best_l1_ratios):.4f}")

Fold 1 - Alpha: 417.3064 | L1-Ratio: 1.00 | R2: 0.5710 | RMSE: 361810.0793
Fold 2 - Alpha: 433.3604 | L1-Ratio: 1.00 | R2: 0.8212 | RMSE: 236585.5723
Fold 3 - Alpha: 460.2542 | L1-Ratio: 1.00 | R2: 0.8490 | RMSE: 226231.0215
Fold 4 - Alpha: 403.2836 | L1-Ratio: 1.00 | R2: 0.7823 | RMSE: 273269.8603
Fold 5 - Alpha: 431.2185 | L1-Ratio: 1.00 | R2: 0.6970 | RMSE: 335975.7361

--- ElasticNetCV K-Fold Cross-Validation Summary ---
Average R2 Score: 0.7441 (+/- 0.1006)
Average RMSE: 286774.4539 (+/- 53699.0062)
Average Optimal Alpha: 429.0846
Average Optimal L1-Ratio: 1.0000
